# Silver Transformation: Electricity Market Data

Cleans and combines Fingrid consumption/production data with ENTSO-E price 
data into a single, timezone-corrected table ready for star schema modeling 
in the Gold layer.

**Input:** electricity_project.bronze.fingrid_consumption, 
electricity_project.bronze.fingrid_production, 
electricity_project.bronze.entsoe_dayahead_prices<br>
**Output:** electricity_project.silver.electricity_combined

In [0]:
from pyspark.sql import functions as F

Creates the silver schema.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS electricity_project.silver")

DataFrame[]

Reads the three Bronze tables into DataFrames.

In [0]:
consumption_df = spark.table("electricity_project.bronze.fingrid_consumption")
production_df = spark.table("electricity_project.bronze.fingrid_production")
prices_df = spark.table("electricity_project.bronze.entsoe_dayahead_prices")

print(consumption_df.count(), production_df.count(), prices_df.count())

35037 35039 32256


Converts the string timestamps into proper timestamp type columns, enabling 
time-based operations like joins and filtering.

In [0]:
consumption_df = consumption_df.withColumn("timestamp", F.to_timestamp("startTime"))
production_df = production_df.withColumn("timestamp", F.to_timestamp("startTime"))
prices_df = prices_df.withColumn("timestamp", F.to_timestamp("timestamp"))

In [0]:
consumption_df.printSchema()
production_df.printSchema()
prices_df.printSchema()

root
 |-- datasetId: long (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- value: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)

root
 |-- datasetId: long (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- value: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)

root
 |-- timestamp: timestamp (nullable = true)
 |-- price_eur_mwh: double (nullable = true)



Adds a Finland local time column, converting from UTC and automatically 
handling the EET/EEST daylight saving transitions.

In [0]:
consumption_df = consumption_df.withColumn("timestamp_local", F.from_utc_timestamp("timestamp", "Europe/Helsinki"))
production_df = production_df.withColumn("timestamp_local", F.from_utc_timestamp("timestamp", "Europe/Helsinki"))
prices_df = prices_df.withColumn("timestamp_local", F.from_utc_timestamp("timestamp", "Europe/Helsinki"))

In [0]:
consumption_df.printSchema()
production_df.printSchema()
prices_df.printSchema()

root
 |-- datasetId: long (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- value: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestamp_local: timestamp (nullable = true)

root
 |-- datasetId: long (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- value: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestamp_local: timestamp (nullable = true)

root
 |-- timestamp: timestamp (nullable = true)
 |-- price_eur_mwh: double (nullable = true)
 |-- timestamp_local: timestamp (nullable = true)



Renames the ambiguous column in each DataFrame so consumption and 
production can be distinguished after joining.

In [0]:
consumption_df = consumption_df.withColumnRenamed("value", "consumption_mw")
production_df = production_df.withColumnRenamed("value", "production_mw")

Joins consumption and production on timestamp using an inner join, since 
both datasets cover the exact same time range.

In [0]:
consumption_production_df = consumption_df.select("timestamp", "timestamp_local", "consumption_mw") \
    .join(
        production_df.select("timestamp", "production_mw"),
        on="timestamp",
        how="inner"
    )

consumption_production_df.count()

35036

In [0]:
consumption_production_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- timestamp_local: timestamp (nullable = true)
 |-- consumption_mw: double (nullable = true)
 |-- production_mw: double (nullable = true)



In [0]:
display(consumption_production_df.filter(F.col("timestamp") == "2025-09-01T00:00:00").limit(1))
display(consumption_df.filter(F.col("timestamp") == "2025-09-01T00:00:00").select("timestamp", "consumption_mw").limit(1))
display(production_df.filter(F.col("timestamp") == "2025-09-01T00:00:00").select("timestamp", "production_mw").limit(1))

timestamp,timestamp_local,consumption_mw,production_mw
2025-09-01T00:00:00.000Z,2025-09-01T03:00:00.000Z,7226.13,5923.28


timestamp,consumption_mw
2025-09-01T00:00:00.000Z,7226.13


timestamp,production_mw
2025-09-01T00:00:00.000Z,5923.28


In [0]:
consumption_production_df.filter(F.col("consumption_mw").isNull() | F.col("production_mw").isNull()).count()

0

Joins the price data onto the consumption/production table using an inner 
join. This also drops September 2025 rows, since ENTSO-E data starts with 15min intervals only from 2025-10-01. This resolves the date range mismatch as a side effect.

In [0]:
full_df = consumption_production_df.join(
    prices_df.select("timestamp", "price_eur_mwh"),
    on="timestamp",
    how="inner"
)

Writes the combined, cleaned dataset as a 
Delta table, ready for star schema modeling in the Gold layer.

In [0]:
full_df.write.format("delta").mode("overwrite").saveAsTable("electricity_project.silver.electricity_combined")

Sanity checks.

In [0]:
spark.table("electricity_project.silver.electricity_combined").count()

32167

In [0]:
silver_df = spark.table("electricity_project.silver.electricity_combined")
silver_df.filter(
    F.col("consumption_mw").isNull() | 
    F.col("production_mw").isNull() | 
    F.col("price_eur_mwh").isNull()
).count()

0

In [0]:
silver_df.count() == silver_df.select("timestamp").distinct().count()

True

In [0]:
silver_df.select(F.min("timestamp"), F.max("timestamp")).show()

+-------------------+-------------------+
|     min(timestamp)|     max(timestamp)|
+-------------------+-------------------+
|2025-09-30 22:00:00|2026-08-31 23:45:00|
+-------------------+-------------------+



In [0]:
silver_df.select(
    F.min("consumption_mw"), F.max("consumption_mw"),
    F.min("production_mw"), F.max("production_mw"),
    F.min("price_eur_mwh"), F.max("price_eur_mwh")
).show()

+-------------------+-------------------+------------------+------------------+------------------+------------------+
|min(consumption_mw)|max(consumption_mw)|min(production_mw)|max(production_mw)|min(price_eur_mwh)|max(price_eur_mwh)|
+-------------------+-------------------+------------------+------------------+------------------+------------------+
|            6317.57|            15553.1|            5764.7|           15474.6|            -10.92|            654.88|
+-------------------+-------------------+------------------+------------------+------------------+------------------+

